In [2]:
import duckdb
import pandas as pd
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

print("Расчет расширенных метрик...")


con = duckdb.connect()

ratings_path = Path('..', 'data', 'raw', 'ml-25m', 'ratings.csv')
movies_path = Path('..', 'data', 'raw', 'ml-25m', 'movies.csv')

print("\nЗагрузка данных...")

ratings_df = pd.read_csv(ratings_path)
movies_df = pd.read_csv(movies_path)

print(f"raw_ratings: {len(ratings_df):,} строк")
print(f"raw_movies: {len(movies_df):,} строк")

con.register('ratings_temp', ratings_df)
con.register('movies_temp', movies_df)
con.execute("CREATE TABLE raw_ratings AS SELECT * FROM ratings_temp")
con.execute("CREATE TABLE raw_movies AS SELECT * FROM movies_temp")

con.execute("""
    CREATE TABLE user_activity AS
    SELECT 
        r.userId, r.movieId, r.rating, r.timestamp,
        to_timestamp(r.timestamp) AS datetime,
        CAST(to_timestamp(r.timestamp) AS DATE) AS date,
        DATE_PART('hour', to_timestamp(r.timestamp)) AS hour,
        m.title, m.genres
    FROM raw_ratings r
    LEFT JOIN raw_movies m ON r.movieId = m.movieId
""")

print(f"user_activity: {con.execute('SELECT COUNT(*) FROM user_activity').fetchone()[0]:,} строк")


# ## Метрика 1: Активность по дням недели

dow_activity = con.execute("""
    SELECT 
        CASE 
            WHEN DATE_PART('dow', date) = 0 THEN 6
            ELSE DATE_PART('dow', date) - 1
        END as day_of_week_monday,
        COUNT(DISTINCT userId) as dau,
        COUNT(*) as total_ratings
    FROM user_activity
    GROUP BY day_of_week_monday
    ORDER BY day_of_week_monday
""").df()

days = ['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс']
dow_activity['day_name'] = dow_activity['day_of_week_monday'].astype(int).map(lambda x: days[x])

print("\nАктивность по дням недели:")
print(dow_activity[['day_name', 'dau', 'total_ratings']].to_string(index=False))


# ## Метрика 2: Распределение оценок

rating_dist = con.execute("""
    SELECT 
        rating,
        COUNT(*) as count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
    FROM raw_ratings
    GROUP BY rating
    ORDER BY rating
""").df()

print("\nРаспределение оценок:")
print(rating_dist.to_string(index=False))

print(f"\nСредняя оценка: {con.execute('SELECT ROUND(AVG(rating), 2) FROM raw_ratings').fetchone()[0]}")
print(f"Медианная оценка: {con.execute('SELECT ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY rating), 2) FROM raw_ratings').fetchone()[0]}")


# ## Метрика 3: Топ жанров

genre_stats = con.execute("""
    WITH genre_expanded AS (
        SELECT 
            userId, movieId, rating,
            TRIM(UNNEST(STRING_SPLIT(genres, '|'))) as genre
        FROM user_activity
        WHERE genres IS NOT NULL AND genres != '(no genres listed)'
    )
    SELECT 
        genre,
        COUNT(DISTINCT userId) as unique_users,
        COUNT(*) as total_views,
        ROUND(AVG(rating), 2) as avg_rating
    FROM genre_expanded
    GROUP BY genre
    ORDER BY total_views DESC
    LIMIT 10
""").df()

print("\nТоп-10 жанров:")
print(genre_stats.to_string(index=False))


# ## Метрика 4: Retention

retention = con.execute("""
    WITH user_first_day AS (
        SELECT userId, MIN(date) as first_day
        FROM user_activity
        GROUP BY userId
    ),
    user_next_day AS (
        SELECT 
            ufd.userId, ufd.first_day,
            MAX(CASE WHEN ua.date = ufd.first_day + INTERVAL '1 day' THEN 1 ELSE 0 END) as returned_next_day
        FROM user_first_day ufd
        LEFT JOIN user_activity ua ON ufd.userId = ua.userId 
        GROUP BY ufd.userId, ufd.first_day
    )
    SELECT 
        COUNT(DISTINCT userId) as cohort_size,
        SUM(returned_next_day) as returned,
        ROUND(SUM(returned_next_day) * 100.0 / COUNT(DISTINCT userId), 2) as retention_rate
    FROM user_next_day
""").fetchall()

print("\nRetention (Day 1):")
print(f"   Когорта: {retention[0][0]:,} пользователей")
print(f"   Вернулись: {retention[0][1]:,} пользователей")
print(f"   Retention Rate: {retention[0][2]:.1f}%")



print("Сводная таблица метрик")


total_users = con.execute("SELECT COUNT(DISTINCT userId) FROM raw_ratings").fetchone()[0]
total_movies = con.execute("SELECT COUNT(DISTINCT movieId) FROM raw_ratings").fetchone()[0]
total_ratings = con.execute("SELECT COUNT(*) FROM raw_ratings").fetchone()[0]
avg_rating = con.execute("SELECT ROUND(AVG(rating), 2) FROM raw_ratings").fetchone()[0]

print(f"\nВсего пользователей: {total_users:,}")
print(f"Всего фильмов: {total_movies:,}")
print(f"Всего оценок: {total_ratings:,}")
print(f"Средний рейтинг: {avg_rating}")


con.close()

Расчет расширенных метрик...

Загрузка данных...
raw_ratings: 25,000,095 строк
raw_movies: 62,423 строк
user_activity: 25,000,095 строк

Активность по дням недели:
day_name   dau  total_ratings
      Пн 53248        3876594
      Вт 52446        3741921
      Ср 50524        3493689
      Чт 49121        3288723
      Пт 49988        3295472
      Сб 48514        3464597
      Вс 50535        3839099

Распределение оценок:
 rating   count  percentage
    0.5  393068        1.57
    1.0  776815        3.11
    1.5  399490        1.60
    2.0 1640868        6.56
    2.5 1262797        5.05
    3.0 4896928       19.59
    3.5 3177318       12.71
    4.0 6639798       26.56
    4.5 2200539        8.80
    5.0 3612474       14.45

Средняя оценка: 3.53
Медианная оценка: 3.5


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Топ-10 жанров:
    genre  unique_users  total_views  avg_rating
    Drama        162519     10962833        3.68
   Comedy        162381      8926230        3.42
   Action        161975      7446918        3.47
 Thriller        161948      6763272        3.52
Adventure        161821      5832424        3.52
  Romance        161068      4497291        3.54
   Sci-Fi        160063      4325740        3.48
    Crime        160855      4190259        3.69
  Fantasy        156809      2831585        3.51
 Children        148391      2124258        3.43

Retention (Day 1):
   Когорта: 162,541 пользователей
   Вернулись: 30,545 пользователей
   Retention Rate: 18.8%
Сводная таблица метрик

Всего пользователей: 162,541
Всего фильмов: 59,047
Всего оценок: 25,000,095
Средний рейтинг: 3.53
